# MCP server 4 — Work orders

Explore the complete read-only work-order MCP contract: discovery, filtering, record details, tasks, costs, plan variance, KPIs, schedules, technician assignments, and failure codes.

**Tutorial contract:** run the tutorial notebooks in order and execute this notebook from top to bottom. The shared CouchDB infrastructure check is demonstrated in notebook 02. This notebook validates its own work-order data dependency through real MCP queries and never exposes or calls a write tool.


In [1]:
from pathlib import Path
import json, os, sys

def find_repo(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "servers").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the AssetOpsBench repository.")

REPO = find_repo()
ARTIFACTS = REPO / "artifacts" / "kdd_tutorial"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
print("repo:", REPO)
print("python:", sys.version.split()[0])


repo: /Users/chathurangishyalika/IBM/AssetOpsBench
python: 3.12.13


In [ ]:
# Load environment variables from .env file in the repository root
from dotenv import load_dotenv

def find_repo(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    return None
    
repo = find_repo()

if repo:
    load_dotenv(repo / ".env", override=False)

## 1. Create a read-only MCP client

`AOB_READONLY=1` is passed to every newly spawned server process before registration occurs. Consequently, mutation tools such as create, update, approve, assign, close, and cancel are not exposed.


In [2]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

MCP_ENV = os.environ.copy()
MCP_ENV["AOB_READONLY"] = "1"

async def wo_request(operation, tool_name=None, arguments=None):
    params = StdioServerParameters(
        command="uv",
        args=["run", "--directory", str(REPO), "wo-mcp-server"],
        cwd=str(REPO),
        env=MCP_ENV,
    )
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            if operation == "list":
                return await session.list_tools()
            return await session.call_tool(tool_name, arguments or {})

def parse_result(result):
    text = "\n".join(getattr(item, "text", str(item)) for item in result.content)
    try:
        payload = json.loads(text)
    except json.JSONDecodeError:
        payload = text
    if isinstance(payload, dict) and payload.get("error"):
        raise RuntimeError(payload["error"])
    if isinstance(payload, str) and (
        payload.startswith("Unknown tool:") or payload.startswith("Error executing tool")
    ):
        raise RuntimeError(payload)
    return payload

async def list_wo_tools():
    response = await wo_request("list")
    return [
        {"name": tool.name, "description": tool.description, "schema": tool.inputSchema}
        for tool in response.tools
    ]

async def call_wo(name, **arguments):
    return parse_result(await wo_request("call", name, arguments))


## 2. Discover and validate the live read-only contract


In [3]:
tools = await list_wo_tools()
contract = {
    tool["name"]: list(tool["schema"].get("properties", {}))
    for tool in tools
}
contract


{'list_workorders': ['site_id',
  'status',
  'asset_num',
  'priority',
  'date_from',
  'date_to',
  'page_size',
  'page_num'],
 'get_workorder': ['wonum', 'site_id'],
 'get_workorder_tasks': ['wonum', 'site_id'],
 'get_workorder_costs': ['wonum', 'site_id'],
 'get_workorder_actuals_vs_planned': ['wonum', 'site_id'],
 'get_workorder_kpis': ['site_id', 'period_months'],
 'get_schedule_calendar': ['site_id', 'date_from', 'date_to', 'group_by'],
 'get_my_assigned_workorders': ['labor_code', 'site_id', 'open_only'],
 'get_failure_codes': ['code']}

In [4]:
EXPECTED_READ_TOOLS = {
    "list_workorders",
    "get_workorder",
    "get_workorder_tasks",
    "get_workorder_costs",
    "get_workorder_actuals_vs_planned",
    "get_workorder_kpis",
    "get_schedule_calendar",
    "get_my_assigned_workorders",
    "get_failure_codes",
}
WRITE_TOOLS = {
    "generate_work_order", "update_workorder", "approve_workorder",
    "assign_technician", "close_workorder", "cancel_workorder",
}
assert set(contract) == EXPECTED_READ_TOOLS, (
    f"Read-only contract changed. Missing={sorted(EXPECTED_READ_TOOLS - set(contract))}; "
    f"unexpected={sorted(set(contract) - EXPECTED_READ_TOOLS)}"
)
assert WRITE_TOOLS.isdisjoint(contract), "A write tool is exposed despite AOB_READONLY=1."
print("Read-only work-order contract is compatible: 9 read tools, 0 write tools.")


Read-only work-order contract is compatible: 9 read tools, 0 write tools.


## 3. List and filter work orders

This first domain query also checks that the seeded `workorder` database is available. `page_size=0` requests all matches, allowing the notebook to select real identifiers rather than rely on placeholders.


In [5]:
SITE_ID = "MAIN"
try:
    listing = await call_wo(
        "list_workorders", site_id=SITE_ID, page_size=0, page_num=1
    )
except RuntimeError as exc:
    raise RuntimeError(
        "Work-order data is not ready. Complete the CouchDB setup from notebook 02 and load "
        "the default data with: uv run python src/couchdb/init_data.py"
    ) from exc

work_orders = listing.get("work_orders", [])
assert work_orders, f"No work orders are loaded for site {SITE_ID}."
print("work orders found:", listing.get("total"))
[
    {
        "wonum": row.get("wonum"),
        "status": row.get("status"),
        "assetnum": row.get("assetnum"),
        "priority": row.get("wopriority"),
        "description": row.get("description"),
    }
    for row in work_orders
]


work orders found: 2


[{'wonum': '1000046',
  'status': 'APPR',
  'assetnum': 'PUMP3',
  'priority': 1,
  'description': 'Bearing wear failure mode on Pump 3 - corrective action'},
 {'wonum': '1000045',
  'status': 'WAPPR',
  'assetnum': 'CHILLER6',
  'priority': 2,
  'description': 'Investigate anomaly on Chiller 6 condenser water flow'}]

In [6]:
sample = next((row for row in work_orders if row.get("wplabor")), work_orders[0])
WORK_ORDER_NUMBER = str(sample["wonum"])
WORK_ORDER_SITE = str(sample["siteid"])
FILTER_STATUS = sample.get("status")
filtered = await call_wo(
    "list_workorders",
    site_id=WORK_ORDER_SITE,
    status=FILTER_STATUS,
    page_size=10,
    page_num=1,
)
print("selected work order:", WORK_ORDER_NUMBER)
print("status filter:", FILTER_STATUS)
print("matching records:", filtered.get("total"))


selected work order: 1000046
status filter: APPR
matching records: 1


## 4. Retrieve one exact work order


In [7]:
work_order_result = await call_wo(
    "get_workorder", wonum=WORK_ORDER_NUMBER, site_id=WORK_ORDER_SITE
)
work_order = work_order_result.get("work_order", {})
assert str(work_order.get("wonum")) == WORK_ORDER_NUMBER
assert str(work_order.get("siteid")) == WORK_ORDER_SITE
work_order_result


{'work_order': {'wonum': '1000046',
  'description': 'Bearing wear failure mode on Pump 3 - corrective action',
  'description_longdescription': None,
  'siteid': 'MAIN',
  'orgid': None,
  'assetnum': 'PUMP3',
  'location': 'MAIN-PUMPHOUSE',
  'status': 'APPR',
  'status_date': None,
  'worktype': 'CM',
  'wopriority': 1,
  'reportdate': '2020-05-02T10:45:00+00:00',
  'reportedby': 'AGENT.FMSR',
  'failurecode': 'BEARING-WEAR',
  'parent': None,
  'taskid': None,
  'lead': None,
  'jpnum': None,
  'schedstart': None,
  'schedfinish': None,
  'targstartdate': None,
  'targcompdate': '2020-05-03T12:00:00+00:00',
  'actstart': None,
  'actfinish': None,
  'estlabhrs': 6.0,
  'actlabhrs': None,
  'estlabcost': None,
  'actlabcost': None,
  'estmatcost': 320.0,
  'actmatcost': None,
  'estservcost': None,
  'actservcost': None,
  'esttoolcost': None,
  'acttoolcost': None,
  'estatapprtotalcost': None,
  'esttotalcost': None,
  'acttotalcost': None,
  'wplabor': [{'laborcode': 'MECHTECH2',

## 5. Inspect child tasks

A work order may have zero or more child tasks. An empty list is a valid grounded result for a parent without task records.


In [8]:
tasks = await call_wo(
    "get_workorder_tasks", wonum=WORK_ORDER_NUMBER, site_id=WORK_ORDER_SITE
)
tasks


{'parent_wonum': '1000046',
 'site_id': 'MAIN',
 'total': 0,
 'tasks': [],
 'message': '0 task(s) under 1000046.'}

## 6. Compare costs and planned-versus-actual values


In [9]:
costs = await call_wo(
    "get_workorder_costs", wonum=WORK_ORDER_NUMBER, site_id=WORK_ORDER_SITE
)
actuals_vs_planned = await call_wo(
    "get_workorder_actuals_vs_planned",
    wonum=WORK_ORDER_NUMBER,
    site_id=WORK_ORDER_SITE,
)
{
    "costs": costs,
    "actuals_vs_planned": actuals_vs_planned,
}


{'costs': {'wonum': '1000046',
  'site_id': 'MAIN',
  'status': 'APPR',
  'assetnum': 'PUMP3',
  'location': 'MAIN-PUMPHOUSE',
  'actual_hours': 0.0,
  'total_cost': 0.0,
  'breakdown': [{'category': 'labor', 'amount': 0.0, 'share_pct': 0.0},
   {'category': 'material', 'amount': 0.0, 'share_pct': 0.0},
   {'category': 'service', 'amount': 0.0, 'share_pct': 0.0},
   {'category': 'tool', 'amount': 0.0, 'share_pct': 0.0}],
  'message': 'Cost breakdown for 1000046.'},
 'actuals_vs_planned': {'wonum': '1000046',
  'site_id': 'MAIN',
  'status': 'APPR',
  'worktype': 'CM',
  'labor_hours': {'estimated': 6.0,
   'actual': 0.0,
   'variance_abs': -6.0,
   'variance_pct': -100.0,
   'over_budget': False},
  'labor_cost': {'estimated': 0.0,
   'actual': 0.0,
   'variance_abs': 0.0,
   'variance_pct': None,
   'over_budget': False},
  'material_cost': {'estimated': 320.0,
   'actual': 0.0,
   'variance_abs': -320.0,
   'variance_pct': -100.0,
   'over_budget': False},
  'service_cost': {'estimat

## 7. Compute site KPIs

The sample records are historical, so a 120-month window is used to include them. A short current window can correctly return zeros.


In [10]:
kpis = await call_wo(
    "get_workorder_kpis", site_id=SITE_ID, period_months=120
)
kpis


{'site_id': 'MAIN',
 'period_months': 120,
 'total_workorders': 2,
 'completed': 0,
 'backlog': 2,
 'overdue': 2,
 'avg_completion_hrs': 0.0,
 'priority_breakdown': {'2': 1, '1': 1},
 'top_assets_by_wo_count': [{'asset': 'CHILLER6', 'count': 1},
  {'asset': 'PUMP3', 'count': 1}],
 'message': 'KPIs for MAIN over 120 month(s).'}

## 8. Build a historical schedule calendar

Explicit dates match the seeded 2020 tutorial records and avoid an empty calendar caused by a current-date default window.


In [11]:
schedule = await call_wo(
    "get_schedule_calendar",
    site_id=SITE_ID,
    date_from="2020-01-01",
    date_to="2020-12-31",
    group_by="date",
)
schedule


{'site_id': 'MAIN',
 'date_from': '2020-01-01',
 'date_to': '2020-12-31',
 'total_scheduled': 1,
 'by_date': [{'date': '2020-04-29',
   'count': 1,
   'workorders': [{'wonum': '1000045',
     'description': 'Investigate anomaly on Chiller 6 condenser water flow',
     'description_longdescription': None,
     'siteid': 'MAIN',
     'orgid': None,
     'assetnum': 'CHILLER6',
     'location': 'MAIN-MECH-CH6',
     'status': 'WAPPR',
     'status_date': None,
     'worktype': 'PdM',
     'wopriority': 2,
     'reportdate': '2020-04-28T09:15:00+00:00',
     'reportedby': 'AGENT.TSFM',
     'failurecode': None,
     'parent': None,
     'taskid': None,
     'lead': None,
     'jpnum': None,
     'schedstart': None,
     'schedfinish': None,
     'targstartdate': '2020-04-29T08:00:00+00:00',
     'targcompdate': '2020-04-30T17:00:00+00:00',
     'actstart': None,
     'actfinish': None,
     'estlabhrs': 4.0,
     'actlabhrs': None,
     'estlabcost': None,
     'actlabcost': None,
     'es

## 9. Find work assigned to a technician

The labor code is taken from the selected record, keeping the example grounded in the loaded data.


In [12]:
labor_rows = work_order.get("wplabor") or []
if labor_rows and labor_rows[0].get("laborcode"):
    LABOR_CODE = str(labor_rows[0]["laborcode"])
    assigned = await call_wo(
        "get_my_assigned_workorders",
        labor_code=LABOR_CODE,
        site_id=WORK_ORDER_SITE,
        open_only=True,
    )
    print("technician:", LABOR_CODE)
    display(assigned)
else:
    print("The selected record has no planned labor assignment; example skipped.")


technician: MECHTECH2


{'site_id': 'MAIN',
 'status': None,
 'labor_code': 'MECHTECH2',
 'total': 1,
 'work_orders': [{'wonum': '1000046',
   'description': 'Bearing wear failure mode on Pump 3 - corrective action',
   'description_longdescription': None,
   'siteid': 'MAIN',
   'orgid': None,
   'assetnum': 'PUMP3',
   'location': 'MAIN-PUMPHOUSE',
   'status': 'APPR',
   'status_date': None,
   'worktype': 'CM',
   'wopriority': 1,
   'reportdate': '2020-05-02T10:45:00+00:00',
   'reportedby': 'AGENT.FMSR',
   'failurecode': 'BEARING-WEAR',
   'parent': None,
   'taskid': None,
   'lead': None,
   'jpnum': None,
   'schedstart': None,
   'schedfinish': None,
   'targstartdate': None,
   'targcompdate': '2020-05-03T12:00:00+00:00',
   'actstart': None,
   'actfinish': None,
   'estlabhrs': 6.0,
   'actlabhrs': None,
   'estlabcost': None,
   'actlabcost': None,
   'estmatcost': 320.0,
   'actmatcost': None,
   'estservcost': None,
   'actservcost': None,
   'esttoolcost': None,
   'acttoolcost': None,
   'e

## 10. Read and resolve failure codes

Omitting `code` lists the reference catalog. When the selected work order has a failure-code identifier, the second call resolves that exact code without modifying the record.


In [13]:
failure_code_catalog = await call_wo("get_failure_codes")
print("failure codes available:", failure_code_catalog.get("total"))
display(failure_code_catalog)

existing_code = work_order.get("failurecode")
if existing_code:
    resolved_failure_code = await call_wo(
        "get_failure_codes", code=str(existing_code)
    )
    print("work-order failure code:", existing_code)
    display(resolved_failure_code)
else:
    print("The selected work order has no recorded failure code.")


failure codes available: 10


{'code': None,
 'total': 10,
 'failure_codes': [{'code': 'FC001',
   'description': 'equipment does not start'},
  {'code': 'FC002',
   'description': 'equipment stops unexpectedly during operation'},
  {'code': 'FC003', 'description': 'abnormal noise during operation'},
  {'code': 'FC004', 'description': 'fluid leak observed'},
  {'code': 'FC005',
   'description': 'excessive vibration, shaking, or instability'},
  {'code': 'FC006', 'description': 'overheating or high temperature reading'},
  {'code': 'FC007', 'description': 'indicator light does not illuminate'},
  {'code': 'FC008', 'description': 'gauge does not operate or is inaccurate'},
  {'code': 'FC009', 'description': 'structural damage or cracking'},
  {'code': 'FC010', 'description': 'part missing or loose'}],
 'message': 'Found 10 failure code(s).'}

work-order failure code: BEARING-WEAR


{'code': 'BEARING-WEAR',
 'total': 0,
 'failure_codes': [],
 'message': 'Found 0 failure code(s) for BEARING-WEAR.'}

## Safety and scope

The server implementation also supports creating, updating, approving, assigning, closing, and cancelling work orders. Those operations are intentionally absent from this tutorial process because `AOB_READONLY=1` prevents their registration. Use a disposable database for lifecycle demonstrations.

## Takeaway

You discovered and exercised the complete read-only work-order MCP contract through real stdio calls while preserving the tutorial database.
